# 2023: Week 3 Targets for DSB
Using SQL to clean and transform banking data.
<br>
The challenge requirements and dataset can be downloaded from [Preppin' Data Challenge](https://preppindata.blogspot.com/2023/01/2023-week-3-targets-for-dsb.html)

In [1]:
# connect SQL to the notebook
%load_ext sql
%sql duckdb:///project.duckdb

Connecting to 'duckdb:///project.duckdb'

### Pivot Targets table

In [2]:
%%sql

SELECT *
FROM read_csv_auto('dataset/Targets.csv');

Running query in 'duckdb:///project.duckdb'

Online or In-Person,Q1,Q2,Q3,Q4
Online,72500,70000,60000,60000
In-Person,75000,70000,70000,60000


Pivot Targets table to create Target_pivot view

In [3]:
%%sql

CREATE OR REPLACE VIEW Targets_pivot AS
SELECT *
FROM (
    UNPIVOT (
    SELECT *
    FROM read_csv_auto('dataset/Targets.csv')
    )
ON Q1, Q2, Q3, Q4
INTO
    NAME Quarter
    VALUE Target)
    ;

SELECT *
FROM Targets_pivot;

Running query in 'duckdb:///project.duckdb'

Online or In-Person,Quarter,Target
Online,Q1,72500
Online,Q2,70000
Online,Q3,60000
Online,Q4,60000
In-Person,Q1,75000
In-Person,Q2,70000
In-Person,Q3,70000
In-Person,Q4,60000


Create output

In [4]:
%%sql
WITH transaction AS (
    SELECT
        CASE
            WHEN "Online or In-Person" = 1 THEN 'Online'
            WHEN "Online or In-Person" = 2 THEN 'In-Person'
        END AS Transaction_Type,
        QUARTER("Transaction Date") AS Quarter,
        SUM(Value) AS Total_Value
    FROM read_csv_auto('dataset/PD_2023_Wk_1_Input.csv')
    WHERE SPLIT_PART("Transaction Code", '-', 1) = 'DSB'
    GROUP BY 1, 2
)

SELECT
    t.Transaction_Type,
    t.Quarter,
    t.Total_Value,
    ta.Target,
    t.Total_Value - ta.Target AS "Variance to Target"
FROM transaction AS t
LEFT JOIN Targets_pivot AS ta
    ON CAST(REPLACE(ta.Quarter, 'Q', '') AS INTEGER) = t.Quarter
    AND ta."Online or In-Person" = t.Transaction_Type
ORDER BY t.Transaction_Type, t.Quarter;

Running query in 'duckdb:///project.duckdb'

Transaction_Type,Quarter,Total_Value,Target,Variance to Target


The query above doesn't return the output so added all query in one cell

In [5]:

%%sql

WITH transactions AS (
    SELECT
        CASE
            WHEN "Online or In-Person" = 1 THEN 'Online'
            WHEN "Online or In-Person" = 2 THEN 'In-Person'
        END AS Transaction_Type,
        QUARTER("Transaction Date") AS Quarter,
        SUM(Value) AS Total_Value
    FROM read_csv_auto('dataset/PD_2023_Wk_1_Input.csv')
    WHERE SPLIT_PART("Transaction Code", '-', 1) = 'DSB'
    GROUP BY 1, 2
),
targets AS (
    UNPIVOT (
        SELECT *
        FROM read_csv_auto('dataset/Targets.csv')
    )
    ON Q1, Q2, Q3, Q4
    INTO NAME Quarter VALUE Target
)

SELECT
    t.Transaction_Type,
    t.Quarter,
    t.Total_Value,
    ta.Target,
    t.Total_Value - ta.Target AS "Variance to Target"
FROM transactions AS t
LEFT JOIN targets AS ta
    ON t.Quarter = CAST(REPLACE(ta.Quarter, 'Q', '') AS INTEGER)
    AND t.Transaction_Type = ta."Online or In-Person"
ORDER BY t.Transaction_Type, t.Quarter;


Running query in 'duckdb:///project.duckdb'

Transaction_Type,Quarter,Total_Value,Target,Variance to Target
In-Person,1,77576,75000,2576
In-Person,2,70634,70000,634
In-Person,3,74189,70000,4189
In-Person,4,43223,60000,-16777
Online,1,74562,72500,2062
Online,2,69325,70000,-675
Online,3,59072,60000,-928
Online,4,61908,60000,1908
